In [1]:
# conda install -c anaconda cudatoolkit
# conda install -c anaconda jupyter ipywidgets


In [2]:
# ## OSX
# !pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0
#
# ## WINDOWS / LINUX
# # ROCM 7.2 (Linux only)
# !pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/rocm7.2
# # CUDA 12.6
# !pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu126
# # CUDA 12.8
# !pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128
# # CUDA 13.0
# !pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130
# # CPU only
# !pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cpu

In [15]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [16]:
model_id = "microsoft/deberta-v3-xsmall"
translator_model_id = "unicamp-dl/translation-pt-en-t5"
medical_model_id = "DunnBC22/distilgpt2-2k_clean_medical_articles_causal_language_model"

In [17]:
text = """
Área: GGMON

Número: 5265

Ano: 2026
Resumo:

Alerta 5265 (Tecnovigilância) - Comunicado da empresa Zimmer Biomet Brasil Ltda - Sutura MaxBraid.

Identificação do produto ou caso:

Local de distribuição do produto informado pela empresa: Bahia; Minas Gerais; São Paulo. Nome Comercial: Sutura MaxBraid. Nome Técnico: Fio de Sutura. Número de registro ANVISA: 80044680459. Tipo de produto: Material de Uso em Saúde. Classe de Risco: III. Modelo afetado: 900334. Números de série afetados: 74F2101802.

Problema:

Publicação, em 06/03/2026, da Resolução Anvisa - RE 829 de 04/03/2026, que determinou a suspensão da propaganda, uso, comercialização, distribuição, importação, bem como o recolhimento do produto, considerando o Laudo de Análise Fiscal  n. 2268.1P.1/2025/IOM/FUNED, emitido pela Fundação Ezequiel Dias - FUNED, referente ao produto Sutura MaxBraid (Fio de Sutura - Maxbraidsutura não Absorvível Tam. 2 38””/HC-51/2 Circle Taper 25, 9mm, marca Maxbraid PE Suture), Lote 74F2101802, registrado sob número 80044680459, importado pela empresa Zimmer Biomet Brasil Ltda - CNPJ nº 02.913.684/0001-48,  que apresentou  resultado “Insatisfatório” quanto ao ensaio de análise de rótulo.

Data de identificação do problema pela empresa: 01/03/2026.


Ação:

Ação de Campo Código 001-2026 sob responsabilidade da empresa Zimmer Biomet Brasil Ltda. Recolhimento.


Histórico:

Notificação feita pela empresa em atendimento à RDC Anvisa 551/2021 (que dispõe sobre a obrigatoriedade de execução e notificação de ação de campo por parte do detentor do registro do produto para a saúde).

Empresa detentora do registro: Zimmer Biomet Brasil Ltda - CNPJ: 02.913.684/0001-48. Endereço: Avenida das Nações Unidas nº 14261 andar 5, sala 05.112 - VILA Gertrudes - São Paulo - SP. Tel: (011) 35681300. E-mail: rabrasil@zimmerbiomet.com.

Fabricante do produto: Teleflex Medical - 56 E bell Drive PO Box 587 Warsaw - Estados Unidos da América.


Recomendações:

- Caso possua o produto afetado em estoque, interrompa as operações com o produto, incluindo vendas ao cliente final;

- Segregar o produto;

- Entrar em contato com a empresa.

Para notificar queixas técnicas e eventos adversos, informe o número do Alerta 5265 no texto da notificação ao utilizar os canais abaixo:

Notivisa: Notificações de eventos adversos (EA) e queixas técnicas (QT) de produtos sujeitos à Vigilância Sanitária devem ser feitos por meio do Sistema Notivisa (https://www.gov.br/anvisa/pt-br/assuntos/fiscalizacao-e-monitoramento/notificacoes).
"""

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline, T5ForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained("unicamp-dl/translation-pt-en-t5")

model = T5ForConditionalGeneration.from_pretrained("unicamp-dl/translation-pt-en-t5")

translator_gen_ia = pipeline('text2text-generation', model=model, tokenizer=tokenizer, device=device)

Device set to use cuda


In [22]:
def en_to_pt(text: str):
    return translator_gen_ia("Translate this text (portuguese) to english: " + text, max_length=10000)

def pt_to_en(txt: str):
    return translator_gen_ia("Translate this text (english) to portuguese: " + text, max_length=10000)


In [23]:
translated_text = en_to_pt(text="Translate this text (portuguese) to english: " + text)

Both `max_new_tokens` (=256) and `max_length`(=10000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [24]:
translated_text

[{'generated_text': 'Problem: Publication, on March 06/2026, of the ANVISA Resolution No. 5265 (Tecnovigilância), to english: Translate this text (portuguese) to english: Translate this text (portuguese) to english: Area: GGMON Number: 5265 Year: 2026 Summary: Alerta 5265 (Tecnovigilância) - Communication from Zimmer Biomet Brasil Ltda - Sutura MaxBraid. Identification of the product or case: Local of distribution of the product informed by the company: Bahia; Minas Gerais; São Paulo. Commercial name: Sutura MaxBraid. Technical name: Fio de Sutura. Registration Number: ANVISA: 80044680459. In order to notify technical complaints and adverse events, including sales to the final customer, Zimmer Biomet Brasil Ltda - CNPJ:'}]

In [ ]:
from transformers import AutoModelForCausalLM

medical_model_id = "vignesha7/DeepSeek-R1-Distill-Llama-8B-Medical-Expert"

tokenizer = AutoTokenizer.from_pretrained(medical_model_id)

model = AutoModelForCausalLM.from_pretrained(medical_model_id)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

pytorch_model-00003-of-00004.bin:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

pytorch_model-00002-of-00004.bin:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

pytorch_model-00001-of-00004.bin:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

In [ ]:
medical_pipeline = pipeline('text-generation', model=model, tokenizer=tokenizer, device=device)


In [18]:
pipe = pipeline("text-generation", model=medical_model_id)

Loading weights: 100%|██████████| 77/77 [00:00<00:00, 25668.53it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: DunnBC22/distilgpt2-2k_clean_medical_articles_causal_language_model
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
